# Create Arnold Ventures Awards

Creates Arnold Ventures (formerly Laura and John Arnold Foundation) awards.
~2,632 grants ingested through the foundation's public Algolia search index.

**Prerequisites:**
- Run `scripts/local/arnold_ventures_to_s3.py` to download and upload the data.

**Data source:** Algolia search index `grants` on Arnold's app
(public App ID + search-only API key)
**S3 location:** `s3a://openalex-ingest/awards/arnold_ventures/arnold_ventures_projects.parquet`

**Arnold Ventures funder (OpenAlex):**
- funder_id: 4320315359
- ROR: (none)
- DOI: 10.13039/100014848
- display_name: "Arnold Ventures"

**Schema notes:**
- `grant_amount` is already numeric USD in the source; currency hardcoded USD.
- `grant_term_raw` is "YYYY - YYYY". Day defaults to Jan 1 / Dec 31.
- `funding_source` distinguishes the four Arnold legal entities (LJAF, ANI LLC,
  DAF, AV LLC) — surfaced as `funder_scheme`.
- `topics_json` preserved for downstream enrichment but not flattened here.


## Step 1: Create staging table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.arnold_ventures_raw
USING delta
AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/arnold_ventures/arnold_ventures_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) FROM openalex.awards.arnold_ventures_raw;

## Step 1.5: Inspect raw + verify amount/currency

In [ ]:
%sql
DESCRIBE openalex.awards.arnold_ventures_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.arnold_ventures_raw LIMIT 5;

In [ ]:
%sql
SELECT
  COUNT(*) AS rows,
  COUNT(grant_amount) AS has_amount,
  ROUND(MIN(grant_amount), 0) AS min_amt,
  ROUND(MAX(grant_amount), 0) AS max_amt,
  ROUND(AVG(grant_amount), 0) AS avg_amt,
  COUNT(DISTINCT funding_source) AS funding_sources
FROM openalex.awards.arnold_ventures_raw;

## Step 2: Transform to award schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.arnold_ventures_awards
USING delta
AS
WITH arnold_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320315359
)
SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.objectID)))) % 9000000000 AS id,
    g.title AS display_name,
    COALESCE(NULLIF(g.grant_description, ''), NULLIF(g.grant_body, '')) AS description,
    f.funder_id,
    g.objectID AS funder_award_id,
    TRY_CAST(g.grant_amount AS DOUBLE) AS amount,
    'USD' AS currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) AS id,
        f.display_name,
        f.ror_id,
        f.doi
    ) AS funder,
    'research' AS funding_type,
    NULLIF(g.funding_source, '') AS funder_scheme,
    'arnold_ventures_algolia' AS provenance,
    -- Algolia exposes start_year/end_year only; default day = Jan 1 / Dec 31
    CASE WHEN g.start_year IS NOT NULL
         THEN TRY_TO_DATE(CONCAT(g.start_year, '-01-01'), 'yyyy-MM-dd') END AS start_date,
    CASE WHEN g.end_year IS NOT NULL
         THEN TRY_TO_DATE(CONCAT(g.end_year, '-12-31'), 'yyyy-MM-dd') END AS end_date,
    TRY_CAST(g.start_year AS INT) AS start_year,
    TRY_CAST(g.end_year AS INT) AS end_year,
    struct(
        CAST(NULL AS STRING) AS given_name,
        CAST(NULL AS STRING) AS family_name,
        CAST(NULL AS STRING) AS orcid,
        CAST(NULL AS DATE) AS role_start,
        struct(
            NULLIF(g.grantee_name, '') AS name,
            CAST(NULL AS STRING) AS country,
            CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
        ) AS affiliation
    ) AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    g.url AS landing_page_url,
    CAST(NULL AS STRING) AS doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.objectID)))) % 9000000000) AS works_api_url,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM openalex.awards.arnold_ventures_raw g
CROSS JOIN arnold_funder f
WHERE g.objectID IS NOT NULL AND TRIM(g.objectID) != '';

## Step 3: Insert into openalex_awards_raw at priority 41

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'arnold_ventures_algolia' AND priority = 41;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    41 AS priority
FROM openalex.awards.arnold_ventures_awards;

## Verification

In [ ]:
%sql
SELECT COUNT(*) AS total_arnold_awards FROM openalex.awards.arnold_ventures_awards;

In [ ]:
%sql
-- Step 6.7 fail-fast amount/currency check
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) AS pct_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies,
    ROUND(MIN(amount), 0) AS min_amt,
    ROUND(MAX(amount), 0) AS max_amt,
    ROUND(AVG(amount), 0) AS avg_amt
FROM openalex.awards.arnold_ventures_awards;

In [ ]:
%sql
SELECT id, SUBSTRING(display_name, 1, 60) AS title,
       amount, currency, start_year, end_year, funder_scheme,
       lead_investigator.affiliation.name AS grantee
FROM openalex.awards.arnold_ventures_awards LIMIT 10;

In [ ]:
%sql
SELECT funder_scheme, COUNT(*) AS cnt, ROUND(SUM(amount), 0) AS total_usd
FROM openalex.awards.arnold_ventures_awards
GROUP BY funder_scheme ORDER BY cnt DESC;

In [ ]:
%sql
SELECT funder.display_name, COUNT(*) FROM openalex.awards.arnold_ventures_awards GROUP BY funder.display_name;